# Piper Voice Batch Generator

This notebook loads location instructions, synthesizes speech files with Piper, and updates each entry with the generated audio file path.


In [ ]:
!pip install piper-tts

In [ ]:
from __future__ import annotations

import json
import os
import re
import wave
from pathlib import Path
from typing import Optional
import piper
from piper import PiperVoice, SynthesisConfig




def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (candidate / "touchscreen-display").exists():
            return candidate
    return start


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

INSTRUCTIONS_PATH = REPO_ROOT / "touchscreen-display" / "public" / "instructions.json"
AUDIO_ROOT = REPO_ROOT / "touchscreen-display" / "public"

OUTPUT_AUDIO_RELATIVE = Path("audio") / "instructions"
OUTPUT_AUDIO_DIR = AUDIO_ROOT / OUTPUT_AUDIO_RELATIVE

VOICE_PATH = REPO_ROOT / "chatbot" / "app" / "voices" / "en_US-amy-medium.onnx"
VOICE_CONFIG_PATH = VOICE_PATH.with_suffix(VOICE_PATH.suffix + ".json")

USE_CUDA = True
SPEAKER_ID: Optional[int] = None
LENGTH_SCALE: Optional[float] = None
NOISE_SCALE: Optional[float] = None
NOISE_W_SCALE: Optional[float] = None
VOLUME: float = 1.0

OVERWRITE_AUDIO = False
AUDIO_FILENAME_TEMPLATE = "{index:03d}_{slug}.wav"


In [ ]:
def slugify(value: str, fallback: str, max_length: int = 48) -> str:
    value = value or ""
    slug = re.sub(r"[^a-z0-9]+", "-", value.lower())
    slug = slug.strip("-") or fallback
    return slug[:max_length].rstrip("-") or fallback


def ensure_dependencies() -> None:
    missing: list[str] = []
    if not INSTRUCTIONS_PATH.exists():
        missing.append(f"Missing instructions file: {INSTRUCTIONS_PATH}")
    if not VOICE_PATH.exists():
        missing.append(f"Missing Piper voice model: {VOICE_PATH}")
    if not VOICE_CONFIG_PATH.exists():
        missing.append(f"Missing Piper voice config: {VOICE_CONFIG_PATH}")
    if missing:
        raise FileNotFoundError("\n".join(missing))
    OUTPUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)


ensure_dependencies()
voice = PiperVoice.load(str(VOICE_PATH), str(VOICE_CONFIG_PATH), use_cuda=USE_CUDA)


def build_synthesis_config() -> SynthesisConfig:
    return SynthesisConfig(
        speaker_id=SPEAKER_ID,
        length_scale=float(LENGTH_SCALE) if LENGTH_SCALE is not None else None,
        noise_scale=float(NOISE_SCALE) if NOISE_SCALE is not None else None,
        noise_w_scale=float(NOISE_W_SCALE) if NOISE_W_SCALE is not None else None,
        volume=float(VOLUME),
    )


voice, build_synthesis_config()


In [ ]:
with INSTRUCTIONS_PATH.open("r", encoding="utf-8") as fh:
    instructions: list[dict[str, object]] = json.load(fh)

updated_instructions: list[dict[str, object]] = []
synthesis_config = build_synthesis_config()

for idx, entry in enumerate(instructions, start=1):
    current = dict(entry)
    directions = str(current.get("directions", "")).strip()

    if not directions:
        current["file_path"] = ""
        updated_instructions.append(current)
        print(f"[skip] No directions for index {idx}; file_path left empty")
        continue

    slug_source = str(current.get("location") or current.get("code") or f"entry-{idx}")
    slug = slugify(slug_source, fallback=f"entry-{idx}")
    filename = AUDIO_FILENAME_TEMPLATE.format(index=idx, slug=slug)
    audio_path = OUTPUT_AUDIO_DIR / filename

    if audio_path.exists() and not OVERWRITE_AUDIO:
        print(f"[skip] {filename} already exists")
    else:
        print(f"[piper] Synthesizing {filename}")
        with wave.open(str(audio_path), "wb") as wav_file:
            voice.synthesize_wav(
                directions,
                wav_file,
                syn_config=synthesis_config,
            )

    relative_path = (OUTPUT_AUDIO_RELATIVE / filename).as_posix()
    current["file_path"] = relative_path
    updated_instructions.append(current)

with INSTRUCTIONS_PATH.open("w", encoding="utf-8") as fh:
    json.dump(updated_instructions, fh, indent=2, ensure_ascii=False)
    fh.write("\n")

print(f"Updated {len(updated_instructions)} instructions.")
